<a href="https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page for one client.

**Tables to be used**: fact_content_daily_performance for the daily panel, and dim_content and dim_clients for joins.

**Time window**:

Decision moment: Single date. As of 2026-03-31. This is the moment the features are allowed to know about.

Feature window: This is 90 days before the decision moment. That is between 2025-12-31 and 2026-03-31. This period mirrors the 90-day trailing convention in the starter CSV.

Label window: This will be 30 days after the decision moment, meaning between 2026-04-01 and 2026-04-30. This period was selected because it is future data that is relative to the decision moment.

**Target/proxy**: a label observed from the features, that is, decline in impressions or page position over a defined future window. Trend_direction and trend_pct will not be included as part of the features.

In [2]:
import duckdb
from google.colab import userdata
con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 0")

Token loaded: hf_IcB... (length 37)


┌─────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │ client_hash_id │ content_hash_id │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_event

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"
month = "2026-03"

# 1. Grain check — zero rows back means the grain holds
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""")

# 2. Row count + date span
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/*.parquet')
""")

# 3. Availability, using IS TRUE per the skill's warning
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘

In [4]:
#Handling the gsc_data_start gap before building features
con.sql(f"""
    SELECT client_hash_id, gsc_data_start
    FROM read_parquet('{rel}/dim_clients.parquet')
    WHERE gsc_data_start <= DATE '2025-12-31'
""")

┌─────────────────────────┬────────────────┐
│     client_hash_id      │ gsc_data_start │
│         varchar         │      date      │
├─────────────────────────┼────────────────┤
│ client_0797ff3a1fc9a6a5 │ 2025-11-05     │
│ client_08a6a72ff48e62c0 │ 2025-09-24     │
│ client_08d2847f24cf89c1 │ 2025-07-21     │
│ client_0e1acc6cd57b0eba │ 2025-09-24     │
│ client_1d09b519bdde7c7a │ 2025-11-05     │
│ client_2094c6eb080311d5 │ 2025-12-17     │
│ client_23a62021009f63c4 │ 2025-09-24     │
│ client_2910fd937f0b4d9a │ 2025-09-24     │
│ client_2c32078d69f2cbad │ 2025-11-05     │
│ client_2e65897d94f60220 │ 2025-11-05     │
│            ·            │     ·          │
│            ·            │     ·          │
│            ·            │     ·          │
│ client_a60a11451483af1c │ 2025-11-16     │
│ client_b10cb2997d0c7c86 │ 2025-06-18     │
│ client_ba65e80a1116ae41 │ 2025-10-13     │
│ client_c182d11e4862a37d │ 2025-06-21     │
│ client_cd12bcfd98942aa1 │ 2025-10-20     │
│ client_d

In [6]:
#feature-window rows count

rel = "hf://datasets/FlyRank/internship-warehouse"
feature_months = ["2025-12", "2026-01", "2026-02", "2026-03"]
feature_paths = [f"'{rel}/fact_content_daily_performance/month={m}/*.parquet'" for m in feature_months]

con.sql(f"""
    SELECT COUNT(*)
    FROM read_parquet([{", ".join(feature_paths)}])
    WHERE report_date BETWEEN DATE '2025-12-31' AND DATE '2026-03-31'
      AND client_hash_id IN (
          SELECT client_hash_id FROM read_parquet('{rel}/dim_clients.parquet')
          WHERE gsc_data_start <= DATE '2025-12-31'
      )
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     22513921 │
└──────────────┘

In [7]:
con.sql(f"""
    SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0
""")

┌─────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │ client_hash_id │ content_hash_id │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_event

In [8]:
feature_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d,
        SUM(ga4_sessions) AS sessions_90d,
        COUNT(*) AS days_with_data
    FROM read_parquet([{", ".join(feature_paths)}])
    WHERE report_date BETWEEN DATE '2025-12-31' AND DATE '2026-03-31'
      AND client_hash_id IN (
          SELECT client_hash_id FROM read_parquet('{rel}/dim_clients.parquet')
          WHERE gsc_data_start <= DATE '2025-12-31'
      )
    GROUP BY client_hash_id, content_hash_id
""").df()

print(feature_df.shape)
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(288815, 7)


,client_hash_id,content_hash_id,impressions_90d,clicks_90d,avg_position_90d,sessions_90d,days_with_data
0,client_62f4a7e64f5e0096,content_ae725e6f1852a254,228.0,0.0,65.287814,0.0,73
1,client_62f4a7e64f5e0096,content_48e8152390b84fa6,14386.0,36.0,3.587036,0.0,91
2,client_62f4a7e64f5e0096,content_de62c7692feb64a6,16269.0,29.0,2.115424,0.0,91
3,client_62f4a7e64f5e0096,content_1a834c5cb078328d,15595.0,12.0,4.179759,0.0,91
4,client_62f4a7e64f5e0096,content_e17c9534314cd3d2,518.0,1.0,10.544365,0.0,90


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.